# Furusato Workshop 3.0.0 — Notebook_04_Furusato_Provision_Complete_Workshop

**Workshop version / 教材版:** 3.0.0

**Runtime profile:** v3.0.0-preview. **Data contract:** 2.7.0-realistic.1 (unchanged CSVs).

**Production placement:** the specified folder directly; no Temp folder is required.

Run one explicit stage at a time. Default is read-only preflight. Native Activator start/stop and correlated complete-file delivery remain separate gates. Run Notebook05 only after the three increments; it reads Lakehouse/Files, not Eventhouse.

Use the same private resource plan, gate and deployment-state evidence across stages. Environment identities are parameters, never embedded in this notebook. CLI deployment is the primary cross-machine orchestration path; this frontend uses normal NotebookUtils authentication. The sealed package includes the source-grounded, complete-contract, time-layer-isolation and standard-contract-restoration compilers, their SQL views and query examples. Apply the corrected profile only to verified fresh sources; the historical four-source baseline alone is not the corrected Agent.


In [ ]:
# Fabric parameter cell — environment identities remain in private JSON files.
PARTICIPANT_ID = "001"
ENVIRONMENT = ""  # Explicitly choose dev, test or prod.
EXPECTED_WORKSPACE_NAME = ""
SCOPE_FILE = ""  # Private scope JSON; use notebookutils.nbResPath + "/builtin/scope.json".
PLAN_FILE = ""  # Private resource-plan.json from preflight.
WRITE_GATE_FILE = ""  # Explicit folder mapping/write approval; never generated as approved.
PRIVATE_EVIDENCE_DIR = ""  # Outside the extracted source package; preserve between stages.
ACTION = "preflight"
APPLY_CHANGES = False
ALLOW_AUTOMATED_APPLY = False
CONFIRMED_PLAN_SHA256 = ""
RUN_NOTEBOOK01 = False
EXPECTED_DEFINITION_SHA256 = ""
RULE_STATEMENT_OVERRIDES = {}


In [ ]:
import base64, gzip, hashlib, json, sys, tempfile
from pathlib import Path
_PAYLOAD_CHUNKS = []


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("fBK5r1pVr2umXm8Y/bv56JhPB3eb2V3BbBh1p603+g0jUB/aX1JsNBlBh93i53fQ4G/4uUl+lwf0w9fawCiwPeg/MDKTvWqt5XfhD3Cex2FxztQzkVMA/fUOuZjqr7w8O5VX4WJ4S9mCz59j+qbMegANPexpcTmXb0GLJSE76RT+2fLIG8RD4eJ7FMwfRjFPHNP/GXo3iqb2vXafYvKFle5bN7z1p+HcMwvvpk254377+n1YvStW2yImf9xItalKquTAdjBWqj7x7LtCoauYc+CAgqr7K9gUrsNeAV69pOlV96Bfpj+e19YYGAq02FWCabf2AD3z8wv1DPXNcPIwJf0AZ32xevh79Rk070EPuV+O6EvwBTDTFDywPqe3Uqz2flwmnh+vIX4O3mPvI9jbkKlOqdDXpOhFs2s22j/MYpaWxxfOouJ1aqqnxjkWaYymGfvLWsDMd+X1FzecGaveXezjOvL91It2VCmpek/u6M5ci3orTentHM6D9LSgY9k7VbLlrlKPbF1z1d6eno/kY0nI5TP6PHh8WAAzke6fl+FQXzwuKaP58/tW8PFI0iEf7sGX34XcGIWR7Ei3e7tP4NIT2SE/fbEm8wPNzj3ydL3fNv2gnpMVCc1TVljHNND442jcloyC6gVsRbhyvCDQXGDdyAlErFuvrxziXE/zHBbYRv3wlMbD6ro7/L4A7ZDOPUFziQ3UneI9rNET6bxr9fhZ2LXasEXkVXc4XnwJLT+mx+R9pyXvC/ZI0Hp6j3P80616Xeu3+RQ0M83wR45uK4sTebJ/NteSPJ9+VdBzz9gqprGtmnXj4bD/yXrMAv1jVZ161Wy39UIf+lczLMOsGi7063IagEZs6NfE5ptP3o9blTC0O4unb8+B2xZ5aEe1ZVVXlepRFzII9C+rsF87en+uKvWV7Y3LWq2A82Qx9CRUo3bQtWzGQMELv8++2s2y/PBzCppYCRpe7u6/OkNd2FAZ/wX//2vniyF/vftddrpVeSnNyQPESPK34Up+2FUTjHSf4CT81KpZX0Nix3n49zHWa9RpWawV+vidts/98Tnhb4/4W4K/vnfwU8IP9PN394THavcP86+P4dPjs//s/IjxFXH3T/60Zt9rxrK9e5nnXoGr7vpG8Yfpm807Y2uADvTvuWerC5z5JfnHQFZwkUd/fjG+Qlid5U+974P3ttBFK80yAwf8qVot2qKR6s1w5n1fVcY74POtqtdLe/7MxzwEvqJ+WXUF/5CXFHcKQUv9sqsUtd2G8Vhg0GFTnE9nCDpotrj/GTv1kXQO+ummZ9OqB9FdM1gMG858dGtC5gl+VX6CZ6m7oN/u1/QbgE+9L3udVpn5NxvhdwobTy2hJ6mPhlO/OzUgK9S1+P+Yt2fQU+5a6LhKuJ7eL6fx+pP3ENc8Hq6L9Rj14JcTXbb8KXvUA2d12nkZsJfBT636qtE3q9nzz+CS8DGkfJ+m38vsQ6PA+xtGYDa+1/x34jbFX3eeH5YPjY5R98/JBb/brBe0pl3o3hUi2+Kc5H7QVeyVaK9XK0IWtMpd+uy9saTplb1cOJwv+LvjVuSHux8866/foVRtN0jPGrKj8vV7E7z9fRXzs+DFfY5Seh/h/7yc97QcF+ZLB7QflyoEkb4wn0ax/5mzn0FHcSPPk5IhS4V5xOa5M+AxVdJ2hYLjzYosGdL4t6gdqlbf6zZ7eD91K3KiSuYKvLiDNgl6C44Ml+Nx4ldm9tFnG+dMxfujJljtoKcS7Dmlb416ajh+7FpNYW/EOG00X7sPrqB/0ubQ9tZTH3ygPIAe1qCNhwS/+eu7m+cvpPnYN5fFYPucsu7Rc6/Nybmyny/hh5P9DlsFO1RllmjZejvkfqvNttvRA4/7bUdVyJ6K7Oh+Cf9GZ/f7ViuOei/OoMtOUT5Dl01VlsOvsnx3n+od0+wrQu8uw5rwDX4HHvt2t/n6/edv8vLLN1z7cGTT7el0j1fjs3NmbtU06+riNLdO+JKNYZLDDZ5sDS3I4zgXGnQsF0eM/YZbnkEIOzNyzuLXHmu8C119XOro9S1tB02pupwV5XjzIB4SbII/1bIa2TsnzjnGWotzyIavjtGWcf91PM792my9zLzKdA/FHu2e5J+uIz/+tOL9on69a8sPX0bpfu51Iq7nfqWYJrZNhR6f4yeKdXF3Ef/e5B6vgSFcufMlvm9H+i3Z+5wP5Zyc4DP+uD3X/K7e8NRo7nYoN3QDe27LTq04ty3T7+jQl5yp2mxJXWW20Q48dWjgAbll0u7uvfc80lmHxglNp+eQYjL8TrykirN4bH6VH92WvBzK8iNx1NH+HX1+tb7gLfK16fh25JcdpQedYUS0dx3wP+Rr3LtA6pVUryLZemVnww57s3xtVOdxPkSfPoiHkXSQsWlpoiEB43RFzOK8j3Yay8gF5equet43Ir7/gl9IyOm8Hf24nj48rsP5IraLBQZZTq/J5dz3a8/T+Pf19bqSt8gNyNVJoFpGCfZYhP1fac2GBxutDP1eJg3bnsNm8Z6jqAV19nY6pY429s2z/1bY8U/FiIczvYLjTmtEuEdPoMH1h2RVUy2pVn2nhfWdmO1s1XHOhkz7TvNau47FUQHGBnxZdLzK1n67PGoLzCXNLayh6Jz6RwA+B9fjFyPhEwyW7dFy9hW//0x8x8KHl9b0ZO9nijjvNZyCs/dZNpvsncirELkGzDHBmY/jXPPj3iolzYNsb7Y22q4YaN4YtlCvoEXqqqsDrgofXa/kQM/ibzvHqwtZhH2AHd5eTkOzRrk+Cm/WTu/qrLW4KVR+n4K57WXtzc/Ys8z9tHN7diQ73oed6ReDjJA1xQeWawQq9kmTVMqVDWT+invGWZW2zn9Zbv9W7Hy+PXPsK8rE96dCZgp/DrCw8oTfV/gd/Mnfq+uYz/1gHSrAxLXHWKYe8WjmfgbzGC7XRLwfP6hFp6lij9phR2+V1GgMnnTm6g74AWIBGDhgbYgKuQssvNXejtku1makMvOTasYO/VIW6vk+JpRhlTeWOF/OU/xD8x3+63/73+dTHjKpTPqtv358SFIJAGFHzan8UFvQNSXTlu5IP77a3RtALl0e/RzL8k4Vn+1V1EnqwTpNJ/jaWZeFefzwpSd3vo5O4O3iXGrVaWoBSKoM+LTRPAPiCmKd7hxoSpKTKC3CBqphG6za2mr6LDqwUOa+U2E2lmfdYZ3pFD+/uKuwG0MVQJSa++US22Tu8Va2KZrr4aBRjFkFolsy5TeK7TJgdrGrjGW8s2R7M1mIba8FcVRlu2VJjSY+IE5B9QABPf9TxLbf8kK3rWTDnv669ZgN61P14brSFVXGzwtXy1/fsA9QTxCzFt59VyjiX8K4naNQBBs7xy3OIU4gknsRTJai1uy9dx/27ZYT+ul5/BG0c6AXwiJP0Iw7/9mliyJ2RRxcHaQ38b3LtDRNv4vrPg36QkhZvUjVWXbX2tki5ONvbeyPrddFD1DbMjaaBVVmtSR79i4T7cx447P8xv2QV93w5oEQc70P61SW3QO0/LmmizO9zirjb7V5V4SMRerVAb6SZ1/l9onvNXufu+dU9vRkh+6apgE5Y28IlexQuHMCW/dXahSEgJ47XCepXi96n/vsZtcDrV/Zx57bHbZ8vMTqyzCAnKoRpjMUSfPi5/pE1cfXfRZdcQ+a7VBVxlHHMiLOw9QU09VqxbkaaQFMXUmLgrmqt31b77vvNP05ItizS+YmKWU7Mf/jMEtTTfVSahKd0U+1vcvAtlqnnwsXzLMsixSGn0l48rzb5YIO/Cz3ga23IsAlT/XoLjTKdBd2wbMdqz13QtGvN4R8L3ebva0Kk3X/XOPQo1SU8TNd07jZMe15gnXYScnpW2Tg5dZ4+25MZyEXz+UTYdelccYiXf+treOyZWy9TAp5e/PpadcJ5Pov/9P/9p/+4/98Hm/RdUp+DisPX6ZSuE5TSw9h2yswn+FdZ7YW4d8aZCj/RhlA/qd7vJZcV1uLVNQ4XOxeCAVflqXLNHU9kyZ8LA9aO00ha6kwI023q7eoZ3ZdyANN6nsdveFqUR8/dqRG9RJdLGdoNnFXJjKvETiWCevA6NflkeTKo92PONyaulQvhE+OPv8IPnstzwJfbO2wLjHVFVispLoMqRiljj7jvyvN6rt2NI7AuVuYibt36ATuU8cZ+C+5pLE/P8PU/XxuX17hKjZhgm4nlpHohFUwfeN+2MCewsyL2K5G3WrUkRHx+TxQJcgwqU4MuwFmLwHPShdk2H5Nb9+rg6v463frN7Eno7BwKsvDwgv08Wp3CmWj4zV8tVYMHYUuqF6k6WPg0VaZuBzvW9QiA/i9jz14Ox61GH49pJiF4c9RsB7pAoOH8nPg2Y/hoi3SpPsq+J32i2vWkxS9o/d21iPZlr4zdeGz9JYDHaFJxobna3t1YMxCQbNacsdqzDWFYYN6WcVZO9Bp4IHC293ejSJ1jWiTtA8pJu+p13dJKkYLOvynbH8pxeEX/L9T0+IURTfEv7K8XP8Wu1VfGWbJ3PMPcZE01aKmT7BHdFH2IkefYa+0sFsrlG2v6gsZoqtRV6+6qj5xnbdjdA2fFYZ1rMdqB1kaSlIJH34uv5GOwkXLe3gYeotl55Fp2fz/RTB8DLxv98sHkfb+7K+/49/RIgy//eaB0Y/Srfd2H747+yxbT9K8igwsXYIcKQJPerT1tJB2PvC7W3ShJGDnV7ZasyWruv1m3tKtm/WkORe44i6sJ/TXLvQWQbYkBbpzav5cWz9Wm336LXWU04GJmdNRkCukm3Pyhn//LJ7r6jPYGHZB8FdY3wp3pDIDz7Vd4EPszxx0BdGgtzaa1Cu+nedg0wzaMf2czM3e44FPmi+ayO1/s661//o//O//5f/8j//p//4P56Ge1XC7mcqx6DYIv/5gxtPe1ZbP/Go+vGCyMCMsjg7l3GvnqnQyzz0zXBIipwyVVHBARqpiyA7ISFV6W4ieDT22IKVIa7Y9OzIghuxixiN9uO+XF9xp0Sh8sG//EO+9CfAzHpgpfIlh0ptEMHhYZ/Yxq5PM0PbGEMGzreYWZFUyBXxVJcfDnsiccPYeEXx2StlpVCTN9qa7LZeRd8a1dvT5Z7nWqnNHsaGiK2Wo8K2mMNuC8NWHOGmVu3eFktoENtfHO9vCv836/rmHcSk+h0Gm4uBtovjSpLV4r073ZOgtZ99Kyxn+f72MFotWFP992VnAfmblFvfTmz38FldpOb+lKu6lPc1c92l76ztQ7VBR4LHivAuVJWCiQje2urXvCtuu3iuqXqUMU5+Rtw/vXz/XzSkDi/OyRpUfwgpd/a9xoSRul/XlcMAb3ClXZNynqclmr2R7lY1jwRwJ+x4rEVWvFUG+7SDT6GaJ1ND0YZZswOXFP9Sl8go3SvZsPsmN8rqJa/821Oz/8z/+v//r//Wf/8N/l1ezmWCmfsvi+oNazSZhx9YAyPM71GeXiRawCrs/c+p0T5JpEexrIlb5Qu+TpCBlVna8GdXqSvX6fpdJQbpJr7Gr4W+O0to4oeM7TXWj6tX5haDvOyNWHwj0fjhiVfVVve2qUW8HdLpT9bEEkSfTQwT0vuneFX3NcoKu4pc1S904zdbbIzX7pn6HKE3iHc/s3bnoy9HnnxZ1gfDW/aIotLHsIpMqIOJ32q7oqkqr0LFUJh4Bbmke//8dcEIEvfOqsdrud7OFSC9GqjLvnok8Xd2jPyJCNdsRYqgSoddMYgGNpkAMu8W5o0/CDiCZJqkyRM68q2i+884EgYtqMCMb4mL3Wkt+cHun8iGvto4//7TkQasH8WpEtqeuYFp6Tg0WHSyjjtWKHKVCK68sBiLpQejorXd4EjNz4FP6yVt12QYln2TZTZaOmJPwt1A5HE/1PP2XeKLi07+MHxarx+F49a/e08Ni37+tnbZGFG3kDjGDm/tszKATnPRiS/uqRbhWtNIRf4/b1+VjEXGrOxeS05s00paJ4joR1+iEhzT68S7pycZO5VbS5iZt7zlQ2YNttG/jJ1rqyXHcxD20BUz+fgwEDm3zwsnzqNlbxy1MT75H8Bu0Zri2UUjWcmgxxP5wVtIfbt8njq0IY4oXe7hPx9m3+OM9BIWIzy+2Sshd359Pa2KPXi41u1pmciHl1BQlJuN8yX5/zNLt/DqOyk52XFNnXwYg9qm2b7k4a90+Hb6bIgb220vzAw77TdoRY3biz9P03eNy6dz1OJuZ2MM/so3BoNbOtd8YsHT6LreO4zI9saakHMeNr6vepPT2Xjp5sbz/L6GX8+Xxr92jzU8jLu0V7SlJLwmP1fbtOfndJA4naC1d24F/2dpz3yqzk7YvSVuWHJdq5b4HzVUW9Ob/gSX4jY1rQPOPboUMa49CDVJdpLG/kr9ax3vEVp6TVN4mmu7HRJKjfvI+2OOik7Y9leQC5OqzU+K47puf1Cj7vpehSO2TYJQXh3dFaKFGkbLaqe3lLpFOFaiF47iD8cBkq8Qo1UwnbUaz8vy2v5tYRtJT02Fap5+U68QGZSzzn4H+BXLuYE2jW3/FFr5sa4b7F1mS0pq93Fvz8eHZZY9HdzH7jQ0zA3cx/Zf58Gk+fTpSb9hWMzCk+Vz1DClu/VkNVH3uM/3A8apz29NC1ZtJACeuxikTnroDJWwZ5lWhau1oVnSamstqG033d1qEH6tX6urGtts0PfzMtbCVqsNwaLGjHrfj6ftBbAHYFPpgyYbeN3vrkWgnWqGxUHCa/VBV7JJj1YvAsTtbn8FAqksAhXPVclx2DtIiG8/rlRyv4XV1hlEqO/wOkD3Hj1HQlL6n6QCnTbxTVhUSqN3d6Hf1+g7CTDeNot5Z9MsTISpaeL9WyY5wH88HSK/sugDrmlcBKJtJXaUF8AVjhWEZZeKrTW3ueHbRUcayqkC0S72SFjGzul7C93Etxyw3/GxnW/HekiCpyBn0CN7WQ3bQKvXWcftXQRKhGjqB40Gh6DaMxcq2q0zCbhNk5fUkO4LBqPckNZqV7SiAhuhtuzrel5m0SnWu6ibOcozvquWuPpYdxd9qulE8s449zAC5SRRb411x3+v0sB6sI2oEKlgBYgiwRY1wvpKm27g/DDnFIEAtq8J3OJNx/q7K6nfPDHFuRVWyN47isMJ6B8NP1izTtT1fPl1PtjtHbkoh+6auJ7favapUtp1QiybxPtHO2aigw26Tmdf1jRqZvur1yjAYAjyvoEot0JEh0d9mS06gKbMSKxxhjIJWG36XBggMNTvslZ2mAcNNfe+6dpl14TltGLaqBN6Bwed4ajTeOB7oV1QINWgYgzZ7eF5LckLQkVIp4lxpBJU1qYdzrGyY5ugwpTFUt07Yit65riizLtcGn4K2PM2rg47sMvg86lrGVvUqpa5Sh8EaeCpD2RJTB9WN5s1DnHEZ5xfaeBs7BH9zfLhSj8ArO/BG+WRdJdHt6NleMIVIZmpO3MH+VsV6tUTm2DR0il12RSBfNY0CaB005ZdBuyWmkOEMIQ9aBUijgiZpvtbs7VSl7WrstmCZoR3i3KxWUWuCwsCTmpdRiUJs15/HrIaq3biA8DQ+I/x/Zj9aeKYR4ft4Xgv73nBFPARyT43UgqpoczsCTSlj0LZfUvU66NsuwDQAPTHE36AhRvlQxAnLXcuG7PBLF/cj9ZGHAeQhoELJFjJPi+auHRkltclnYt+b4HGvLmtRlc4RGevyaRB2lUaAdyzg/MDf6hbP5Y8Y2Y49ce0Q61ImgR2qO+1U5kWEbp3cZM6bHSujs2uB3AJ/tiHzydu2DBkmqeRxXayl6Fg2ZFG9YOO5oE/QDVOtZ5Em9V2m+5FWcK6QT/O5pk88mDi+c3UtN7nnO7rBs6D/O4IshfxobUFv9ItDxjiBLZHmVMrWnSapOJv2nCnhoBNZxbNsnAsrRaEnQsjnUGs6rrq79Hwap/0cPTDGpQq5YMhOs15kJYfmmXP8HRAD+2LZEuSwrymtSKSxSG1XbbJbxExi2gvOYANDHbzOKj/IGMWWL9HDKO+vZ8vuDH/MPepCPNOHrpMgx0B7kJ084yYr2SA/9NZWhXZTJQN8O+N6Rcov9HER6y9DV0Anks/6ftdqgEbUS/yRG+KWo4dmq6hGONemWnQYpmIllYe9gcwEfpB53lpUwbpUPN+WeUbibLCHvM72AHkk6itG0eplTcGKTs6jOBfwqnaT/FtcjhYaOxx7SdUFZNm8EMM10HqTcboW+Lbvs5MH1+iE7Tn4eMO0SduyN1rTAE1Q1kIfK9WQOtFRAmID6IGebEdcD/ZIr+Mcs3JjtR3Vcp3mG0KWNuMOzYMiW6Obm0HJ3nW88VqtyY9jmP6Cjym7LHUHetiAZ6B/55BhnHgCOQcs41hqZIeQV4pRxIky/brAydaabroxvTfAY23QGuQudCp0tpTt7j/ad9BfRZ3CocJwEFW2uc7DRdGBe44zvdcP0PdedBa+k5fjW+iNmN5lyHfKH88mpvL6kHsGaAtyV7ehF+o4P+gLpvApoN3Q2GqQjbbVoI4tgM9k7KvH7k0iNuq1ysRpGXpf7zuUldTnwwSAFfRS+z7nWuFEptsgt179MK1gPRxU5REwpZARFs9uVmCKA2Qz/u1jzxysaQYZDn0PjORAVgq9aWEfoxhvaHp96+iNAHoW8pppEtC8kPk2f3av2uco3+GZZgw7o7J7aXEzapr8/QfMmmdOdZjm9xr0CswGzIiz3diQ56QLyC3Qh13oKhXo/B70bb1M3GtLNnRdNdTCHvEKsIwP2q7OnZBYrzqHjNxpeUxwZa/7ub1OYljC2TZgp9tB9ckZ9IN0EvCwebPphMFmasluzHN9V2MqF/k5BI+FLeg6Ypc+8Cj5itjG2KmwI2wJ2AqYCjoBWAx6IuxBR4t9DjVljLOoAEvM8M71DM8d9ni8uzl0tSv+vtEuTn3Q7o2BGVfKmTfzUdi/7/l+jk6EbrJ6Za7DETocdg7j+KHQ81uV4xUi4B/YDsBbwB3gT+B2De+lRv6GWAD0BFxM3Qy+9XBmGV12MCv958zvK4cO31oROtsB5oinmXU40JFd8fFvOoYhtndmhW7TLqjQ9dCbMvTNhqUDmkIeg63hqVs7xN4B+0BWMN1+I3A1cB3WWxLr9TQX+AOYBIYkru/evbzG8d5sVVejLJ8dbBDYFXQ4A/t5PrAocI7iF5gHoMKUp24TZ6+3gE/xmQ58r/slyAjgFciOJmyhpiHxnaALQuionR3Wdxl6fT7wCXGI/WwU5lXapOxYZ+zkwvQ2tglxT5yLCZ3L0h8D+mYM3KGBp4FJge1hxc3VEDzvTVx+xmshI7CnY5wfdRJkLdPmFZwjUzWpQ2vn92gc9sOOdODxjkgPu3kaSRO5E96URrFLCe9tgp6pY/sh7GbglTroCnYo1oA9g/yHDczghTKDnDSApdi4q7UBltlCI5d4rtDTZeLZroX/0yuFa3vjHFLXV5n1rrnehEcDYJMCJ0KK9FvIZFUPQuBKiRWY2DvI7jrwaQB92AONz9iVBPK9Al1IbDCWmJ7dZd4P+Bv6aavm9GJmH0p4f/cm3PPlrphZr1yYNGN9qFqUCSZ4rQe5C7kMDKFaOCsGN0A3WC/1xYbdgIEKJMbpbR32O+wPyHTsC3PAYXfjepX60wP2zvBfMi0wrphd9Fa5aUvM61j0d/i9kOU3m10raPfoE8h94ITIgP1eEb4N0HEJNvSOmBtyGpgcKBjYz5ZMyF0npD0H3QE+nABP2Thr2OPKJIcvx82bkjNou8O7i9Nq2AF8wxTDzk4+TBISriDwVqQFMb37wHmwZRUTz23wecB/7VDQXciSrX5A2wn2IvgNmBv2JG099pjDfxKDqrBX6EtxM3S1EVMGwvayc2lSwU6+A3Z4Hi96z/uOErCbEhdb1ClCz0tBBHkN3dfn4OY0FPGc0THKcOAs7cJ2bruFtANkpIb13FlAnkAXALNGBvcUZ60FohxNojzziU2ASzTwGuxW2AHAFyUtpG4AbcI+BIZ36R+CXgJmN2BLQr+67zwLiZMYgnWSh/EM2lrwnYB7QaPEbEFIO1mjf0nHurx6fM2d/HMspSN0ZljzxAMOjATeaPYpt8FL9FVVII+YCtiHvYL3pN0NjAcMAoQOWWk57FIGjMI8O38LrAq9ChsKGucw3SieAGGH/vPlsBTwe/NmR39ATv+UqsTWUQdYD88EVmU+WxCowl6DXSu1tp2SsxyX6KZVn9Vmn76tDegbWJu+CcoI4Ca9JbDDxGo85c5ScoD3A7y7seFgenbjwZnRngUOoHaCfQ8JbVvEiQ2PNjR4m6UjxRhv+rBlx0UN1wHjSNwfp/bnnWX6TuPd76zgF11BVAtynfmPoQ1Zrc27dzfScKABz2yyvjHoXT8SuoXdAxX6CWclgSsgszSR8DCmL7MMOQ/7eIa9boOuhS6P+HxhP0Z9nDdTweuQd0bpvTybGwfl3mwYhsQ7QXbMcY5GAXJEhs0DzAG7GjJO3b/TFrRB3AP7m/4gyaS+kkRnKuiQPW3UsvRO/T6fU9dCNgGDUzbWI8h7YETsteIDiwfsiEO8XqZk0hTIfgV8Qt0Dfu0qc+JHlvGCPthRyP9L6F3T2R1V82PfSt/ToiAkfeB9ztO7Av3Oc/NmkD3gZ9hwsG/pF9mpkoo1Qx6zRJS6PYTdSwwH3tfow9Ebc+4LsD9tVehn2vbATX8NvUu0+VQdMpVl9SF4PmwAj8wKmnuO3ul/g80IHaiBhgX2i8ZF6MaN2hQ+KWDSNnBHT7KZmt9s++QLNYSOCOk3nJUcIcthU4jv0Hc8K/419A67S8feKZC9Huxb4Q+FPtfb/nl6r0iqMhHXCj5RcApej3SM9wMW140S5BrWIOxTyH3oK30MzArZKBG3E6/Td9kTZfzgFxm08pfQO84rYomOprT4mcx9sPWKrFnqWXrvNtuwm/uQ3wZwBbvWQUbq80CLGGOwS/RT0XdJvlAFTQAxKX5EPR533QHNSLTD6CsE/qV/4+6voPcb6OQ6aFAlz0EntzYaaAOYw7fPynfSRUD/At6/wgQdWQuJ/8G3oFuNsRbaFhb9cZUysQi7Q9I/KnxN+niHc3AFVlNg6wNjQr/9NfQOHEa/FDvZaLTPKW/xPpTP5+ldLWCvZGJSR+m7sEMi6KqIshG0AHnnuNT1NpPasO/A6CzVgb1clznZFzIiZPyCuEFt9rY4d/rbPpfe039zWAT62uqBtrU5/dk27H/ytBb1NixNxTqhr6CnLMgk6DjsA32+HmiX/pYNkxNBrzhjYDZ2+mz2GKr9HFpN/55dr+dTf3pd0K3TbBMLgY78EpPhQMM+6xCwVuArgamAgWmjTHAm7LxX2ZEHYzzcE7FI+jE+CzsdJopm6YK2rL1TWboG+xH2iSv0e8hc6zowMGz+CLgxGsNmUDesrdAi7Cd1JbC88AFbdcpS2FU2MAPtxMrn0MWi6g4Hif+HiaVC3sKubbZElyHacbBFZHaEw7ky1oT18z2o7xsu45eaSDiE7JQYWwJd6PRpjzfE5PZb6eBkCunvayYjTppMcygkMQ8mjPrYO/ATeA14geco4kqgR+gogzo5YNcp6J45dHWpK+i4BdnKrrWQy/RteizDrONcIMfdj67zaF+lw9zLzqK/chIbH+dXZvsdxhpox2A9zB8Adu4xJg5Zy/JvyEsL2KoJ/Q/cx3iORqykz2FbE3O1dprUh46EblGgq+4+e+25FgFRdv2Mnzjs+guaJJ5hci5khkQ6iMt6qx6xkaOrsk1/YpOtVGj7N0LYn2zLwBga9qEOdMu2DGrxzf6AF9cfT6jLrht2VgGyixEyv6u0oG2xx5BZsT3JeKYfqdSzjKsq0EWQKeBZyZaYz8B8BcgTC3aYhTtYQOYK7NbaJ6+7dDTB7VZdT5pFP/ahsW1A3+0Kvytsfg+yV8RRwQMK9cRYUknvSp1d1cinRTU0qH+xx33oOpZMAb81jRKvUa028cbujXpV30+QLCR7fCz/RHoa9Ms+1UkujKz+U2Lvl2Cf74SvCDurMQap1GHDzEjfZdFNuQncTwzGJF7GJz3mIPjQMT50dou6d0d7SOV7hy3cpfVGOfjCSGD3Jp2w/Jyt4cnSUhdY3gFtOIxfSr2dGk2gk8Zlxnnos4UtDrsAv0v0iAH3R+BpSS0x6kysGft/hf8a+AZnRzvns+VPKZ3MuZ9Yl6UlWdhTnkEbpkAcBhoqkD+7lEecaEHPOHNQJMj+aD7nmdG21dhmhbELizZbfaN64AeF/iYn+ONoKVM7ucvQU0jZUwdNVD2NeJJddb1eGXKJ+U8sBSdepa2xFfkJFnOD6swvYUwMNAVsqjA+ZOK7sF0Zd9H9P4yesgn0eXqC/vGYs8T4MGyFphmokQkbgrxBe0OjzQ/5z5gFCwToH9Q8xiE1YAtHwfp1xt4D2Ng4C+i/T6cngRO0VbzvzLug/IHOpY8VuEtli6MIvKqDbpoqeKDNwo6Ifh3Y62zvILN4RdOBwZpYO3AjbVsV69d02l3G2/ZdrKcS6yTdYcwcawGmD9vUoYHGfDLKlqgRMhYFbMX9K1Jvgp7ptwB+ZNupFmwxm9P2isSUbL0Cm9bL6dRM/I91jB2pXRyF/Sfn7mYuUiCTiY6wZZ9tYL7JbXtu5/LTYFvAfrSJO7Em7AdzsKBv6kU18ov4HvAL8KlOPz/QIKdowPbB/gDzsXWUSn9iwFwiTWLMGzhHUrPxj804FHYL9TVTZZlLtHaa26AjNZadwX4aqYc1L0bNGxc22wrvwkKUYjzpV+j5JK+vjXNqe6rUgD4HpsO+qiKn0KZeAX+NJYetlaDTOR0ZsrnIDqmwa+Q4FgJ7SuhKQ4JcYLw4oMz76Hp5DpAPTzkfckh/aUWiDI19KuAN+uCYA0VfOTtxe+MN8Cj4v808O/Aa/Yvk9QnogLE3lf5El9hA1UWrvCwt7ka09YjpRXrtjTuxZH/UxFlb/mqcTicNgfMkkQsRp77WwTtN8w7X6kmBltwRuRKxDxQ4qOSIvBjIU4u8Mo5s5lyAozUFGCoEjpboX4G+g3x29DnsAOYs2YyzbqBZGGeBjQ3do4OP9Kr3zv1lPsYzW6ixS/ZR7bXqQKdxonBm7bAVaceylRnsgAi2Ozvfejbslj6wtsjnkUX+kef4WpM2dGVrR4y/Togdy5rIO4S0wPXA4rCFKrtPWPudPZj8sBdaMF44SV5fnzrkCbrmkHMiqTuVMUbuGdbOqU5ijREnCvQk5nBARxRB3z5s3l0Xck3Qv6IBlzNXgjLWZl7KTsSQQuDwd8oK1qdORK7vTebv5r0t3qe3Ht/6ib3LdnmQTYy7Y0/ZSs5m9/4mc4lAE151rjFepDAGB9mri7xa8AHkGOnGgs7Te3hXYBYddgbzft3PXfOw1ADersoxzmORorBfRCYW/QsCQ0AnMN9E2O3KHL+zTRt9Xdh/6m52oPYaoAnGAargx7rkMI8mcubggXfyJLDFrSn0b/bvg8RGAG1spnEssQCeCuP2KJQdzNkQOTIbVeQJQKZHZqCFbQ9yzxU8Sv71jBLsM9FJm3qafiX6RmzWSDf7/jvpmsVjRYF9Spm/mzfZuHG67g11qMjhCpkjQ/8A+E7kO4r47Ia5sQ5sehE34HQf6m1vDl3Tg14xIb9bnDwbdXVN5AVwGu0nrzsC/+3pOcaXBvQh259VysADzA/A3k2wv8CWEWM6NvMPiNHKzC/UaPM3aU9CVws5aQubmDEx5jW8m54zOR6TprmKfQ1GVndvHGHntmStSRu8ziLMrcNcWr0BvAGJbfFfxv6wVvr1FAd2WpszsaD3Z5TdRGu+RoQU6835O/c3LbW7kIPCfEycN/tveFXsZ6Voh0aJ7X8d5qhyD4nvRdyKbRGh24lBWLcQGpDJLdpnzM0Abp4Db/Q9vFPpvWulbGb++UDa533tc+XHkBexrICdrfvQ0TPsDZ7JfFbG54VMYF5IBXvN3BWT9jfx45b+HdDpDrxGP3HE6XFdywCWGkecyAe6eZ+sKGnQcb+zqPz+KNdUHpSSMredHKW6xKE9EYp8Sklt2jvbqm9w7i5w07zLrvqeL9GHI3zbjOURB3sB5APoVYLuiLjXjTgOQH8rdGg2z+YtdMz97ESVndrI5bXPJ7fsvdFmWQzwnbbX4WrIdqPEmtg3CfjSspkPDDlCOVCXRT6aRb8UczY47cEoUIfjbLbMUXIgcyAbC2KqTBSA1uu7j9GJuhnE+cX3zPsG/nh2ijfRhHmOzQA6Xc3QTD1yYK/SilDZlZ8YDpgEfFlU6a+k/mO9CWPqEek+YDswSIk6dDzOyZvTxyBj7+e2R5xb371bvyQ0g304pRmpjZ/Az+w7dS8wgylsNuBoyO55AIzHnhoh9CLrRQLQhohNMNeVWIU5uqDtHWNENs4LNhVofwxbFxjQy9VfvGPfDex7Nsf9ZM3M9ykL/64OPcx4rQLMFxn0Qc6BS+b0R8N+lZhfAj4oQe+xlkj4p1TIE9YfQaMAD85KNnPO8rUs71iznV9zSeQtBadrB41boBGsnfEA3CdUhV+Jvif61MF7kS9yrtlqD3aVFMeDemXGELpN+mpNYQdpwkfQKhMrvm7tKa8WaVcv8exokosVVekz94n9bdrTREpYF/EOa3NYq6Rx2lVIf6RdYt4A9DR4zy85OutluH7WUKn0pAUOfffR+FU0nM8hP/WvqyJPz2Bc2qctheeE8bQ/6GTgddpPtg47q6luqKdhN7F9cUCfr6oDSzQNCfYMroH8JkZl7vnrMHKc788if0sudArChoo6zLkNnSXeQXw+kJLPMzEhh/Er+hdJexbjk8bW0SmbKM8gH/RgLnLfLZvtR0HDgSfypoT/J4DNxZgFzjmk74C5nKCTT1rzGHJ1Urt5nAw05v59m0qiRHdNzCxyzK32fCyZN/fJvdIGAbmYF30+zK2L7LgeiT/KBHoRGLnJhiisPWR9G2Qya24gD4n1IEu9LluJhi1qoDlrgEBfEeywrVP71Pd7Yqn+sIm1A5/sfXeD1be41FbOfH79XZmXq1E/KoyD9OeajjOjP15nzqNP7Aq6ZD3WjHUuO5W+fU6LU2YR6+7op+8qPqeKSORb0KDcvftL3xX6YsI88QLsjYzvgvWRY4n1QWxVzVxPYdN7k7mw7RTmtTFf32cevh/XdZoh8/NVxivwX5cx84i2Cifm9iTYU++QATfNuOaUeflyOLTGsJkg1xe9HD6ellai3L6TyW29HxTie6V5jbn8Po0YuBzHvTXgHdhMnsiLBk7uQ17MZOo68CKQcX3L2kTQM2y/GccQyMl74oxxbdPYaPrc0yL1U99vXOrzmnv67ZwwWORyzb0JsQH4jfmH9HWB9yTYV+Q9fe6rwEm2BB7TsXbWzAB/MieX/n5iWuCzuSPqgRxRQ2GzLmD3KXS4mVhtYAztAfoO6/49l1+n6RXKAMqJUOTYMc+AOaVKP9CAHegDYCYSa5JgK4YiJ17qu07YD5mboMKu6oraA5u5bND/vVdiO2GPrBL/xXOfvjrY3b10b0tV7MEW5zVnDdU/qzzcn42oWWMjFiuuheoM0pjM4Z2zn1/nJV/k6zus226yXhk6lnkjii9pIfOSbNbjbiFFyrCRgCvBR01jx3oEyIktc/5FrF6qSyLnRq/63Xys4928NL41If+wV03WCG7nQ8l8gJ0K/VlMYx8Z2ZH5fFC8YY7NpKTh//u5nBRgHfA/8857rIfewn70iC+xftbDwM5t+PTVd3WR2824Fem5BN0PXS9yurEPDU/kswAfQb+Hf8j5/pPKx9yZ0vYFT8OOT/Yg+w6s/9fmWtTmxPatw7yQSGM9Pm0V5qtj3ZpP/hIxBE+FDG2xRphYFnrLhMyBXGS9I/1owOm29yYcuxkOqoFoDnV3imVZHwjMX6a9ZXsm/p+xTNaLV0V+kEM/mPCPOSL23FWwHp3+BtJMg7kB0L/MyxoXYBP4Il/zLTJ8UIU9aMbtKkqZ3+N6C+HnzcaJiJtVkYun8dki9mJHEzGCBnY+aNn0hB/EsguqLtrCl7vi/5lX0ZKFPyeyd5AHRYe5tNH4df4b4a/bBtMz9gDw/paYmH4l/B4xb6mL9XV15qDQ3+jTny459HnAhsKaoF9mHGvCsSdyt9mmnC3jHGAn1rfsHaC+bg/p44D8mD0bwD3gw3I/7u8oan8z+1bUJObAizp82WashzlUTfpgGZckD7Vkp4n16rBnuWdN2NEcvcI6O+b0cQyNpcp8L7xLRJ/Y6/aNNj94gqO0Er13IQZYZJ2XI3JGiU/HZdK/qDXmeJLI33WFjcUcd6xf1I5CZotYK6+bh4xnak0tsFnHwV4JuvnuNWKfN5MBZK40gZxqz0dXa4gN1tcyRl3CPRj7oE8Lspa1buAjnT4to0RfAji/aEvMXTTpnSkImaCAJr0qa2ZhW4MXm/RxvA6nToTtDPu9tpdVWGe1OLlVn4fQIazvAiYuAVM8s19CJg8V8lRjDA02qhGJXEzWDzf7wnZmfq1qsQaDMrMCfWgANzBnrAo8BEzRNDjhu8C8IBtYTxPf63sf3+9VEZjzuWMFa/bFnSStdcaZ+j2HeZjQ66rF3DXQschVot0DvmHdi0J9z1r5cZk1uQ7Ht0GmqfRh4D001qzxTGhHKWPYGHbhlbqPExUh/yHzU/0HfAldALslb/NciR27os+BwBrgSYujLfAf7ROOcxD9MSpF+vWZRwW9HXDiqKOwNgK0xBiKN2ePaU6Od4W801/pc7m678U5380ZHNe3GdhXv6w1VeZKFWy9zdxM0C9rq/oiNwN8J7E2DHoO78aeLBOPPZ3Zm4STcJlPpiqzHeg/slmj9+69PsSq9nTfSGv89/EeEW+lBxk2smtHVdFdhnm5jBMCB8jMuRf2JPAf/Zua+G8SsNYzrplv0K4BLsSaJWD+sPFuuj4bk/da7OewYc9r4POtGjbYa4BVTawNiiBv2QdnBzslgn0fipxzS90Cv4DuBd4T+dCiNwV9NiH71dSvyYuNPQggxxrYywvnHE088cyIMRna0DOZ8t6BvcCePJrOnEbuJ/Qqc84kx+9a7KXRE+N6oNcKrNWGbINUIc1g3zI+hYnFOlhtPmpq904mZ4VYPeXxQ3y0xf4xW9a8ij4pETHrTCaGFfknzKtSZiJminVDp5GfDOA+6DErqZtlz6ZQ3XTZj4WYSsnuj8OGn8LW6Fjbpb2APbXIxbfYx6IwIWYqqXnbkfiHoxDY34d2b5NjNrAuibFZ+kD7zN0BHgce8GBXito70xU1enh/hz2eILtAY3PmZOH7mwxtSWxUG7dmKwIHtXciLyJb612KJzMbkpA5dWfgANvdbOKePVkaG5dErNuz2bOjzDo6YFCX+SrE0ALzswdTBH7VW+wHtGUfa/KBw3WLGgP2nxD9jph3xfrkN+3hJGw8TSxTdyhfiv2jnlStLZ/L2AjwhKxBBtvM55YM0DTz6OmvUZlHJ9H2hm4qixxI2C+q1QPus0EDzFVljBZ7bIk69zfvZfoOOb8TcBL9pLbIhZkzz4x1CqUu8RF1nkLegK0Fmncon0XedN9XPY5WYR173PeME3xVkV8NGk3aFiZtHN1xUgvL0ZBJC7e7UWki2l92gbdVpZK0eUviaoWDP9/CWfM6VcThK0IWqLX4vsNMy8DjHmkv9r8CPtk/V/RS2a/tXtDZoh8NIjzL3fdzKeL6TaYHTCZOkvT/Kv6Oex760GhcK69P2xKCbiaFTN+pBvRBiW1IK1st816se3MG1QLkYTJSytzvF5sj79cd9ougv6XojWQe7oXPv9/3Hl7d8m72EEwu9XOFodqn0jQS4z3XMxWMAHC4DCDMRAASAED0NR2U8KIZUL+/LsyPaYbCu3Jt0uCoeANizhiBYXuHQ/oKJbNgh2IYomlnXhgC2v04bGSNHBcKbD0Njbg/6+3lz8clc8PNZILTxTUttOL4tgoGM3dJgPKF9ScJ3Jff4SyhHq0RgGErOYPWepxtgp91YkvBE4tNjvd3evaaeF/P3V/sUePGo8BwpJvd+e+LrsQZ4QhwcOsEY1c+BGjdM9872buz9zjsh3tp7Xmmy7xH3P83R4vGSZ/e48DsyfdP9yf5+yEgnQzreFsP4LPPOepTnO9DvA/cx6Av03SjVI3w/S1ojwk29+S5ibUtZO8jrlng84DNbLYUOIXcmhdiLphIuh6RP84JQAiqIelN9NHM9jAG4MN5JNenXbIFuMoldhdy17UPfUHjBhP4nnHozdp+ZpGDnUnsHRycxPeJIf08Su+xYO/Yvhw3Woh73LJPqrPvzdxOjcOczNIzTlZ2G48TOW5gPDYWBOWt29Qxe9jL1HmdVTid3UMs6HP7EtNkEuwxRgVBF+6hk/vbepEmf7wkmwHwg0Mv1oUddbzKWq0V2ZBMAp+lMsRNmns8gCYsyrf9dzwV15dTIOEC/AOEi07zjdy9vdZa2529zviTG6ImA3P2ThXu7be9nHgt0Dh/rePENLxl8zg6WvIKP6O4Y5rK9hh+K+C49p3dO74THX3nqB8y3hc6Jdtj2QhvSgI00SGTe082mDA35Pl0QFTmWhaIZEHQRQBy6FndpkEbjBNZp7pHvX3Ba47U8KYGe/xuCX4eeV7JZIzkmoMTPX/2FbfjlrOO92fKStB4vEec737qgEt5dQ1jnHo43o9mY0eHPZuzQI4AdLFHcS/fI75pZpMhCzQsj4zlvJyELhoOmIBsNvAsP06A3PdB9o97Ldt4BxiPor/xRGCtuKFax4JB0dw+T4WDoM05dUXo7DShVDgWnNvEWRcA8Fta0thUyNVoDD5KjXUnhH7BO+KeXIM3OcjkJDFPGA21qcECnEZxdJuRX/t1ApcleiFJuGZvaNGsOGmolAv+gT6gX0T/ZgYWsN+NaCL6RBc9nt2YvfxrhZ2qtCQ2UtgH6EEz2Atv2BT9riO9NMHZJo3smjfeJO6DHXUGcfM17CV0QsGtLQrfXytjn8YPy+m/TrfDcBlMDwK2EAvOpkmFVLhLOiz/uOVQ5wa7LC5AIP4PEJPSE0ReAgB9HJeqjNpT0dHq+z3fVWmCQ+t9waFpMXHEiwYQ8ZKJRvEzmmZZdF4aZD6/K4poUuZ5IUEECP3wLCja00M+fiYr4fvzUUODYnWgnHAgfuDHgnsrqphZXTFaQEjQotwVk+e0cplumXXEE5hiQLXmOJmk28NGvTtZq2YDIFLJmoe1ZokrXqNoYt6OJ9X5AcBLse0kSnzQK8CyweH+u/8fQNlPPxo7CwE=")


In [ ]:
_COMPRESSED = base64.b64decode("".join(_PAYLOAD_CHUNKS), validate=True)
if hashlib.sha256(_COMPRESSED).hexdigest() != "3d5ec95cca5d7a6da7133265e6acb65d30abd9df16a036b6ecdf063a193bc21f":
    raise ValueError("Embedded package fingerprint mismatch; no authentication attempted.")
_FILES = json.loads(gzip.decompress(_COMPRESSED))
_ROOT = Path(tempfile.gettempdir()) / "furusato-preview30-3d5ec95cca5d7a6d"
for _relative, _encoded in _FILES.items():
    _path = (_ROOT / _relative).resolve()
    if _ROOT.resolve() not in _path.parents:
        raise ValueError("Unsafe embedded package path.")
    _raw = base64.b64decode(_encoded, validate=True)
    _path.parent.mkdir(parents=True, exist_ok=True)
    if _path.exists() and _path.read_bytes() != _raw:
        raise ValueError("Existing package bytes changed; use a clean driver session.")
    _path.write_bytes(_raw)
sys.path.insert(0, str(_ROOT / "tools/provisioning"))
sys.path.insert(0, str(_ROOT / "tools/data-agent"))
for _module, _directory in {'preview30_runtime': 'tools/provisioning', 'preview30_deploy': 'tools/provisioning', 'preview30_verify': 'tools/provisioning', 'preview30_ontology': 'tools/provisioning', 'preview30_agent': 'tools/provisioning', 'preview30_activation': 'tools/provisioning', 'preview30_compatibility': 'tools/provisioning', 'v3_artifacts': 'tools/provisioning', 'workshop_runtime': 'tools/provisioning', 'reference_kql': 'tools/provisioning', 'activation_runtime': 'tools/provisioning', 'source_grounded_profile': 'tools/data-agent', 'answer_contract_profile': 'tools/data-agent', 'time_layer_isolation': 'tools/data-agent', 'standard_contract_restoration': 'tools/data-agent', 'fresh_grounded_profile': 'tools/data-agent', 'native_evaluation': 'tools/data-agent', 'native_mcp': 'tools/data-agent', 'measured_contract_profile': 'tools/data-agent'}.items():
    if _module in sys.modules and Path(sys.modules[_module].__file__).resolve().parent != (_ROOT / _directory).resolve():
        del sys.modules[_module]
print("Sealed portable package loaded. No Fabric mutation has occurred.")


In [ ]:
if ACTION not in ('preflight', 'deploy-sources', 'verify-sources', 'deploy-realtime', 'deploy-core', 'handoff-ontology', 'verify-gold', 'deploy-semantic-model', 'refresh-model', 'verify-model', 'deploy-agent', 'publish-agent'):
    raise ValueError("Action is outside this notebook's bounded role. Use the CLI for importing notebooks.")
from pathlib import Path
from types import SimpleNamespace
import json
import shutil
import preview30_runtime as rt
import preview30_deploy as deploy
import preview30_verify as verify

context = notebookutils.runtime.context
if ENVIRONMENT not in ("dev", "test", "prod") or not SCOPE_FILE or not PRIVATE_EVIDENCE_DIR:
    raise ValueError("Set explicit environment, private scope file and private evidence directory. No cloud call made.")
scope_document = rt.load(Path(SCOPE_FILE))
scope = rt.scope_from_document(scope_document, ENVIRONMENT)
if scope["participantId"] != PARTICIPANT_ID:
    raise ValueError("Participant parameter differs from the private frozen scope.")
if scope["workspaceId"] != context["currentWorkspaceId"] or scope["workspaceName"] != context["currentWorkspaceName"]:
    raise ValueError("Notebook context differs from the private frozen workspace.")
if not EXPECTED_WORKSPACE_NAME or EXPECTED_WORKSPACE_NAME != context["currentWorkspaceName"]:
    raise ValueError("Set the exact expected workspace name before execution.")

class NotebookCredential:
    def get_token(self, scope_uri):
        audiences = {
            "https://api.fabric.microsoft.com/.default": "pbi",
            "https://storage.azure.com/.default": "storage",
            "https://kusto.kusto.windows.net/.default": "kusto",
            "https://analysis.windows.net/powerbi/api/.default": "pbi",
        }
        if scope_uri not in audiences:
            raise ValueError("Unsupported token audience.")
        return SimpleNamespace(token=notebookutils.credentials.getToken(audiences[scope_uri]))

credential = NotebookCredential()
evidence = rt.private_directory(Path(PRIVATE_EVIDENCE_DIR))
client = rt.EvidenceClient(scope, evidence, credential=credential)
current_notebook = client.request("GET", "/workspaces/" + scope["workspaceId"] + "/items/" + context["currentNotebookId"]).json()
if current_notebook.get("folderId") != scope["folderId"]:
    raise ValueError("The current Notebook is outside the explicitly selected folder.")

if ACTION == "preflight":
    if APPLY_CHANGES:
        raise ValueError("Preflight is read-only; APPLY_CHANGES must be False.")
    prefix = "/workspaces/" + scope["workspaceId"]
    inventory = {
        "workspace": client.request("GET", prefix).json(),
        "items": client.paged(prefix + "/items"),
        "folders": client.paged(prefix + "/folders"),
        "recoverableItems": client.paged(prefix + "/recoverableItems"),
    }
    baseline = rt.load(rt.PREVIEW / "provisioning/baseline-hashes.json")
    for relative, expected_sha in baseline["files"].items():
        path = rt.BASE / relative
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
            raise ValueError("Extracted baseline byte mismatch: " + relative)
    state_file = evidence / "deployment-state.json"
    owned_state = rt.load(state_file) if state_file.exists() else None
    plan = rt.build_plan(scope, inventory, baseline, owned_state)
    rt.save(evidence / "resource-plan.json", plan)
    rt.save(evidence / "inventory-preflight.json", inventory)
    RESULT = {"mode": "read-only-preflight", "planSha256": plan["planSha256"],
              "blockers": plan["blockers"], "folderMappingVerified": scope["folderMappingVerified"],
              "cloudMutationPerformed": False}
else:
    read_only = ACTION.startswith("verify-") or ACTION == "preview-metadata"
    if not read_only and not APPLY_CHANGES:
        raise ValueError("No write made. Review the private plan, then explicitly enable APPLY_CHANGES.")
    if not read_only and context.get("isForPipeline", False) and not ALLOW_AUTOMATED_APPLY:
        raise ValueError("Automated mutation is not explicitly enabled.")
    if PLAN_FILE:
        external_plan = rt.load(Path(PLAN_FILE))
        rt.save(evidence / "resource-plan.json", external_plan)
    args = SimpleNamespace(
        environment=ENVIRONMENT, scope=Path(SCOPE_FILE), evidence_dir=evidence,
        write_gate=Path(WRITE_GATE_FILE) if WRITE_GATE_FILE else None,
        confirm=CONFIRMED_PLAN_SHA256, run_notebook01=RUN_NOTEBOOK01,
        expected_definition_sha256=EXPECTED_DEFINITION_SHA256,
        rule_statement_overrides=RULE_STATEMENT_OVERRIDES, credential=credential,
    )
    actions = {
        "deploy-sources": deploy.deploy_sources, "deploy-notebooks": deploy.deploy_notebooks,
        "verify-sources": verify.verify_sources, "deploy-realtime": deploy.deploy_realtime,
        "deploy-core": deploy.deploy_core, "preview-metadata": deploy.metadata_change,
        "apply-metadata": lambda a: deploy.metadata_change(a, apply=True),
        "handoff-ontology": deploy.handoff_ontology, "verify-gold": verify.verify_gold,
        "deploy-semantic-model": deploy.deploy_semantic_model, "verify-model": verify.verify_model,
        "refresh-model": deploy.refresh_model_once,
        "deploy-agent": deploy.deploy_agent,
        "publish-agent": deploy.publish_agent_candidate,
    }
    if ACTION not in actions:
        raise ValueError("Unsupported action; no private endpoint fallback.")
    RESULT = actions[ACTION](args)
print(json.dumps(RESULT, ensure_ascii=False, indent=2))
